In [45]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
root_path = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.append(root_path)
from theOne import theOne
import pandas as pd
import numpy as np
from core.data_sources.clob import CLOBDataSource
from core.data_structures.candles import Candles

In [46]:
root_path

'/Users/schalkvisagie/csHonours/project/25349589-MN8-src/quants-lab'

In [47]:
# Load Candles
clob = CLOBDataSource()
CONNECTOR_NAME = "binance"
INTERVALS = "1s"
trading_pair = "LAYER-BNB"
# DAYS = 60

clob.load_candles_cache(root_path)
all_candles = clob.get_candles_from_cache(CONNECTOR_NAME, trading_pair, INTERVALS)
print(all_candles)

candles: Candles = all_candles
candlesdf = candles.data

# candlesdf
# filtered_df = df[df['quote_asset_volume'] > 0]
# print(filtered_df)

2025-06-02 14:49:10,042 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x302e19910>


In [48]:
def load_market_data(connector_name: str, trading_pair: str, data_type: str = "order_book") -> pd.DataFrame:
    """
    Load market data from files for a specific connector and trading pair.
    
    Args:
        connector_name: Name of the connector (e.g., "bitmart_paper_trade")
        trading_pair: Trading pair symbol (e.g., "LINK-USDT")
        data_type: Type of data to load ("order_book" or "trades")
    
    Returns:
        pd.DataFrame: Concatenated DataFrame containing all data from matching files
    """
    folder = root_path + f"/data/order_book/"
    
    # Define the pattern based on data type
    pattern = "order_book_snapshots" if data_type == "order_book" else "trades"
    
    # Find all matching files
    files = [
        file for file in os.listdir(folder) 
        if connector_name in file 
        and trading_pair in file 
        and pattern in file
    ]
    
    if not files:
        raise FileNotFoundError(f"No {data_type} files found for {connector_name} {trading_pair}")
    
    # Load and concatenate all matching files
    dfs = []
    for file in files:
        df = pd.read_json(folder + "/" + file, lines=True)
        dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)

# Example usage:
order_book_df = load_market_data(CONNECTOR_NAME, trading_pair, "order_book")
# trades_df = load_market_data("binance", "POL-USDT", "trades")


In [49]:
order_book_df.rename(columns={"ts": "timestamp"}, inplace=True)
order_book_df

,timestamp,bids,asks
0,1748862149,"[[0.0011898, 345.73], [0.0011897, 3890.64], [0...","[[0.0012041999999999999, 1487.03], [0.0012044,..."
1,1748862150,"[[0.0011898, 345.73], [0.0011897, 3890.64], [0...","[[0.0012041999999999999, 1487.03], [0.0012044,..."
2,1748862151,"[[0.0011898, 345.73], [0.0011897, 3890.64], [0...","[[0.0012041999999999999, 1487.03], [0.0012044,..."
3,1748862152,"[[0.0011898, 345.73], [0.0011897, 3890.64], [0...","[[0.0012041999999999999, 1487.03], [0.0012044,..."
4,1748862153,"[[0.0011898, 345.73], [0.0011897, 3890.64], [0...","[[0.0012041999999999999, 1487.03], [0.0012044,..."
...,...,...,...
5224,1748868540,"[[0.0011979, 343.39], [0.0011978, 5168.22], [0...","[[0.0012178999999999998, 1466.24], [0.001218, ..."
5225,1748868541,"[[0.0011979, 343.39], [0.0011978, 5168.22], [0...","[[0.0012178999999999998, 1466.24], [0.001218, ..."
5226,1748868542,"[[0.0011979, 343.39], [0.0011978, 5168.22], [0...","[[0.0012178999999999998, 1466.24], [0.001218, ..."
5227,1748868543,"[[0.0011979, 343.39], [0.0011978, 5168.22], [0...","[[0.0012178999999999998, 1466.24], [0.001218, ..."


In [50]:
candlesdf
order_book_df

# Ensure timestamp columns are of the same type (int)
candlesdf['timestamp'] = candlesdf['timestamp'].astype(int)
order_book_df['timestamp'] = order_book_df['timestamp'].astype(int)

# Merge on 'timestamp'
candles_and_ob_df = pd.merge(
    candlesdf,
    order_book_df,
    on='timestamp',
    how='inner',  # Only keep rows with matching timestamps
    suffixes=('_candle', '_orderbook')
)

# Display the merged DataFrame
candles_and_ob_df['datetime'] = pd.to_datetime(candles_and_ob_df['timestamp'], unit='s')
# candles_and_ob_df.head()
candles_and_ob_df

,timestamp,open,high,low,close,volume,quote_asset_volume,n_trades,taker_buy_base_volume,taker_buy_quote_volume,bids,asks,datetime
0,1748862149,0.0011975,0.0011975,0.0011975,0.0011975,0,0,0,0,0,"[[0.0011898, 345.73], [0.0011897, 3890.64], [0...","[[0.0012041999999999999, 1487.03], [0.0012044,...",2025-06-02 11:02:29
1,1748862150,0.0011975,0.0011975,0.0011975,0.0011975,0,0,0,0,0,"[[0.0011898, 345.73], [0.0011897, 3890.64], [0...","[[0.0012041999999999999, 1487.03], [0.0012044,...",2025-06-02 11:02:30
2,1748862151,0.0011975,0.0011975,0.0011975,0.0011975,0,0,0,0,0,"[[0.0011898, 345.73], [0.0011897, 3890.64], [0...","[[0.0012041999999999999, 1487.03], [0.0012044,...",2025-06-02 11:02:31
3,1748862152,0.0011975,0.0011975,0.0011975,0.0011975,0,0,0,0,0,"[[0.0011898, 345.73], [0.0011897, 3890.64], [0...","[[0.0012041999999999999, 1487.03], [0.0012044,...",2025-06-02 11:02:32
4,1748862153,0.0011975,0.0011975,0.0011975,0.0011975,0,0,0,0,0,"[[0.0011898, 345.73], [0.0011897, 3890.64], [0...","[[0.0012041999999999999, 1487.03], [0.0012044,...",2025-06-02 11:02:33
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5063,1748868379,0.0012115,0.0012115,0.0012115,0.0012115,0,0,0,0,0,"[[0.0011986, 3739.54], [0.0011985, 343.22], [0...","[[0.0012178999999999998, 1476.5], [0.001218, 2...",2025-06-02 12:46:19
5064,1748868380,0.0012115,0.0012115,0.0012115,0.0012115,0,0,0,0,0,"[[0.0011986, 3739.54], [0.0011985, 343.22], [0...","[[0.0012178999999999998, 1476.5], [0.001218, 2...",2025-06-02 12:46:20
5065,1748868381,0.0012115,0.0012115,0.0012115,0.0012115,0,0,0,0,0,"[[0.0011987, 343.16], [0.0011986, 3739.54], [0...","[[0.0012178999999999998, 1476.5], [0.001218, 2...",2025-06-02 12:46:21
5066,1748868382,0.0012115,0.0012115,0.0012115,0.0012115,0,0,0,0,0,"[[0.0011987, 343.16], [0.0011986, 3739.54], [0...","[[0.0012178999999999998, 1476.5], [0.001218, 2...",2025-06-02 12:46:22
